In [1]:
# This code simulates a single neuron of choice
from brian2 import *
import sys
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
sys.path.append('Tools')
from gaussianGenerator import *
from plottingTools import *


import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, FloatText, Dropdown
from ipywidgets import interactive as interactive_ipyw

In [2]:
# Create sliders for parameters
timeStep_slider = FloatText(
    min=0.0, 
    max=1.0, 
    step=0.01,
    value=0.5, 
    description='Simulation Time Step:', 
    continuous_update=False, 
    readout_format='.2f',
    style={'description_width': '150px'},
    # layout=Layout(height='40px', width='400px')
)
neuro_timeStep_slider = FloatText(
    min=0.0, 
    max=1.0, 
    step=0.01,
    value=0.5, 
    description='Neuron Time Step:', 
    continuous_update=False, 
    readout_format='.2f',
    style={'description_width': '150px'},
    # layout=Layout(height='40px', width='400px')
)
tau_slider = FloatSlider(
    min=0.01, 
    max=10, 
    step=0.01, 
    value=5, 
    description='Tau Value:', 
    continuous_update=False,
    # layout=Layout(width='400px')
)
amp_slider = FloatSlider(
    min=0.01, 
    max=10, 
    step=0.01, 
    value=1.0, 
    description='Amp Value:', 
    continuous_update=False,
    # layout=Layout(width='400px')
)

widgets = {
    'timeStep': timeStep_slider,
    'neuroTimeStep': neuro_timeStep_slider,
    'tauVal': tau_slider,
    'ampVal': amp_slider
    }

In [3]:
# Parameters
Vth = -48*mV
V_reset = -80*mV
input = False

In [4]:
def interactive_simulator(timeStep, neuroTimeStep, tauVal, ampVal):
    NeuroGrp = NeuronGroup(1, LIF_sim_eq, threshold='V > Vth', reset='V = V_reset', method='euler', dt=neuroTimeStep*ms)
    defaultclock.dt = timeStep*ms
    NeuroGrp.tau = tauVal*ms
    M = StateMonitor(NeuroGrp, 'V', record=True)

    if input:
        num_neurons = 120
        gaussian_gen = Gaussian_Input_Generator(num_neurons, amp=ampVal, mu=180, sigma=5, noise=False, normalised=True)
        inputGrp = SpikeGeneratorGroup(1, [0], [5]*ms)
        input_weights = gaussian_gen.generate_cue()
        
        connecting_input = Synapses(inputGrp, NeuroGrp, model='W_input : volt', name="input_synapses", on_pre="V_post += W_input")
        connecting_input.connect()
        max_val = input_weights.max(0)
        connecting_input.W_input = input_weights.max(0)* volt
        network = Network(NeuroGrp, inputGrp, connecting_input, M)
    
    else: # feed dirct current I_ext
        I_ext = ampVal * volt *10**-2
        NeuroGrp.I_ext = I_ext
        network = Network(NeuroGrp, M)  
        print("I_ext: ", I_ext) 
             
        
        
    # visualise_connectivity(connecting_input)
    
    BrianLogger.log_level_error()
    network.run(30*ms)

    figure()
    plt.plot(M.t/ms, M.V[0])
    plt.plot([0, 30], [Vth, Vth], 'r--')
    plt.legend(['V', 'Vth'])
    plt.xlabel('Time (ms)')
    plt.ylabel('v')
    plt.show()
    
    # print("Amplitude of the input weights: ", max_val*(10**3), "mV")


In [5]:
ui = VBox([timeStep_slider, neuro_timeStep_slider, tau_slider, amp_slider])
out = interactive_output(interactive_simulator, widgets)
display(HBox([out, ui], layout=Layout(align_items='center')))